# Inferencia sobre Hold-Out Set y Extracción de Predicciones (Dataset Telco)

Este cuaderno tiene un objetivo estrictamente operativo: cargar el modelo preentrenado (`RoBERTa`) desde el entorno de producción (Google Drive), levantar sus tensores a la memoria VRAM de la GPU, y realizar inferencia ciega masiva sobre la matriz retenida del dominio Telco.

En este caso, la ejecución extrae predicciones probabilísticas para los **3.081 tickets** de prueba (`test_set_telco.parquet`). El CSV resultante no contiene métricas; es un listado puro con la topología predictiva: `[Texto del Ticket | Clase Real | Clase Predicha (56 Clases) | Confianza Probabilística]`. Este artefacto es la materia prima matemática para la Fase 6 de evaluación financiera y cálculo del SLA.

In [1]:
# Instalamos fastparquet
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.4 MB/s eta 0:00:00


In [2]:
# Celda 1: Aprovisionamiento y Carga del Modelo Preentrenado
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

# 1. Montaje del sistema de archivos en la nube
drive.mount('/content/drive')

# 2. Definición de rutas absolutas
BASE_DIR = '/content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR'
TEST_DATA_PATH = os.path.join(BASE_DIR, 'data/gold/test_set_telco.parquet')
MODEL_PATH = os.path.join(BASE_DIR, 'modelo/produccion_roberta')

# 3. Ingesta del dataset de evaluación (Hold-Out ciego)
print("Cargando dataset de evaluación...")
df_test = pd.read_parquet(TEST_DATA_PATH, engine='fastparquet')
df_test['full_text'] = df_test['full_text'].fillna("").astype(str)

# Nota: No reajustamos el LabelEncoder. El mapeo id2label ya está integrado en el config.json del modelo.

# 4. Descongelación del modelo en la memoria de vídeo (VRAM)
print("Instanciando Tokenizador y Modelo desde almacenamiento local...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# 5. Bloqueo de gradientes y envío a GPU
model.eval()  # Modo evaluación estricto
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Arquitectura cargada correctamente. Modo inferencia activado en: {device}")

Mounted at /content/drive
Cargando dataset de evaluación...
Instanciando Tokenizador y Modelo desde almacenamiento local...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Arquitectura cargada correctamente. Modo inferencia activado en: cuda


# Tokenización, Batching y Extracción de Probabilidades Máximas

Para procesar los 3.081 tickets sin saturar la memoria de vídeo (VRAM), convertimos los datos al formato `Dataset` nativo de HuggingFace y aplicamos tokenización dinámica. Usamos la clase `Trainer` en modo inferencia (`do_predict=True`) como mero orquestador para inyectar los lotes en la GPU.

Finalmente, las activaciones en crudo de la capa de salida (*Logits*) se normalizan a través de una función `Softmax`. Extraemos la probabilidad más alta (la confianza máxima del modelo en su propia predicción). Esta confianza es el parámetro crítico que dictaminará si el ticket se resuelve automáticamente o requiere la intervención manual de un agente humano (Bajo el umbral operativo del 0.85).

In [3]:
# Celda 2: Tokenización, Inferencia Batch y Exportación de Resultados
import numpy as np
from scipy.special import softmax
from transformers import Trainer, TrainingArguments

print("Transformando DataFrame a formato Dataset de HuggingFace...")
dataset_test = Dataset.from_pandas(df_test[['full_text', 'target_tripleta']])

print("Aplicando tokenización estricta (max_length=256)...")
def tokenizar_lote(batch):
    return tokenizer(batch["full_text"], padding="max_length", truncation=True, max_length=256)

dataset_test_tokenizado = dataset_test.map(tokenizar_lote, batched=True, batch_size=1000)
dataset_test_tokenizado = dataset_test_tokenizado.remove_columns(['full_text', 'target_tripleta'])

print("Instanciando Trainer estéril para gestionar lotes en VRAM...")
args_inferencia = TrainingArguments(
    output_dir="/content/tmp_inferencia",
    per_device_eval_batch_size=32,
    report_to="none",
    do_train=False,
    do_predict=True
)

trainer_inferencia = Trainer(
    model=model,
    args=args_inferencia
)

print("Ejecutando inferencia neuronal sobre la matriz ciega...")
salida_prediccion = trainer_inferencia.predict(dataset_test_tokenizado)
logits_test = salida_prediccion.predictions

# Protección de estructura en caso de tupla devolviendo logits
if isinstance(logits_test, tuple):
    logits_test = logits_test[0]

print("Aplicando Softmax y vectorizando decodificación de clases...")
probabilidades = softmax(logits_test, axis=1)
clases_predichas_id = np.argmax(probabilidades, axis=1)
confianzas_maximas = np.max(probabilidades, axis=1)

clases_predichas_texto = [model.config.id2label[id_] for id_ in clases_predichas_id]

print("Ensamblando DataFrame de resultados operacionales...")
df_resultados = pd.DataFrame({
    'full_text': df_test['full_text'].values,
    'target_tripleta_real': df_test['target_tripleta'].values,
    'prediccion_roberta': clases_predichas_texto,
    'confianza': confianzas_maximas
})

RUTA_RESULTADOS = os.path.join(BASE_DIR, 'results/predicciones_holdout_roberta_telco.csv')
df_resultados.to_csv(RUTA_RESULTADOS, index=False)

print(f"Exportación de telemetría finalizada: {RUTA_RESULTADOS}")
print("Entorno Cloud liberado. Transfiere el CSV al entorno local para iniciar la Fase 6 de evaluación de negocio.")

Transformando DataFrame a formato Dataset de HuggingFace...
Aplicando tokenización estricta (max_length=256)...


Map:   0%|          | 0/3081 [00:00<?, ? examples/s]

Instanciando Trainer estéril para gestionar lotes en VRAM...
Ejecutando inferencia neuronal sobre la matriz ciega...


Aplicando Softmax y vectorizando decodificación de clases...
Ensamblando DataFrame de resultados operacionales...
Exportación de telemetría finalizada: /content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR/results/predicciones_holdout_roberta_telco.csv
Entorno Cloud liberado. Transfiere el CSV al entorno local para iniciar la Fase 6 de evaluación de negocio.
